# Week 6 Exercise - Product Pricer 

This notebook implements a simple product pricer and evaluator for Week 6.
It uses an OpenAI model to estimate prices and compares them against a small ground-truth dataset.

In [ ]:
# imports
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any

from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import gradio as gr

In [ ]:
# setup OpenAI client
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"OpenAI API Key exists and begins {api_key[:8]}")
else:
    print("OpenAI API Key not set - pricing calls will fail.")

client = OpenAI()
MODEL = "gpt-4o-mini"

In [ ]:
# simple product dataset
@dataclass
class Product:
    name: str
    category: str
    brand: str
    cost: float
    true_price: float


PRODUCTS: List[Product] = [
    Product("Wireless Mouse", "Electronics", "LogiTech", 10.0, 19.99),
    Product("Mechanical Keyboard", "Electronics", "KeyPro", 40.0, 79.99),
    Product("Running Shoes", "Footwear", "FastFeet", 30.0, 69.99),
    Product("Noise Cancelling Headphones", "Electronics", "QuietCo", 80.0, 159.99),
    Product("Coffee Maker", "Home Appliance", "BrewMaster", 25.0, 59.99),
]

In [ ]:
SYSTEM_PROMPT = """You are a pricing assistant for an e-commerce site.
You estimate fair selling prices for products based on their cost, category, brand, and market norms.
Always respond with a single JSON object of the form {"price": <number>} and nothing else.
Prices should be realistic and in USD.
"""

In [ ]:
def price_product(product: Product) -> float:
    """Call the LLM to get a price estimate for a single product."""
    user = (
        f"Product: {product.name}\n"
        f"Category: {product.category}\n"
        f"Brand: {product.brand}\n"
        f"Cost: {product.cost}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]

    resp = client.chat.completions.create(model=MODEL, messages=messages)
    content = resp.choices[0].message.content

    try:
        data = json.loads(content)
        return float(data["price"])
    except Exception:
        # fallback: try to extract a number
        import re

        nums = re.findall(r"[0-9]+\.?[0-9]*", content or "")
        return float(nums[0]) if nums else float("nan")

In [ ]:
def evaluate_products(products: List[Product]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for p in products:
        pred = price_product(p)
        err = abs(pred - p.true_price)
        pct_err = (err / p.true_price * 100.0) if p.true_price else float("nan")
        rows.append({
            "name": p.name,
            "category": p.category,
            "brand": p.brand,
            "cost": p.cost,
            "true_price": p.true_price,
            "predicted_price": pred,
            "abs_error": err,
            "pct_error": pct_err,
        })
    return pd.DataFrame(rows)

In [ ]:
# quick batch evaluation
results_df = evaluate_products(PRODUCTS)
results_df

In [ ]:
# optional Gradio UI for single product pricing

def price_ui(name, category, brand, cost):
    p = Product(name=name, category=category, brand=brand, cost=float(cost), true_price=0.0)
    pred = price_product(p)
    return pred

with gr.Blocks(title="Week 6 Product Pricer - Victor Barny") as demo:
    gr.Markdown("## Product Pricer\nEnter a product and get an estimated price.")
    with gr.Row():
        with gr.Column():
            name_in = gr.Textbox(label="Name", value="Wireless Mouse")
            cat_in = gr.Textbox(label="Category", value="Electronics")
            brand_in = gr.Textbox(label="Brand", value="LogiTech")
            cost_in = gr.Number(label="Cost", value=10.0)
            btn = gr.Button("Estimate Price")
        with gr.Column():
            out_price = gr.Number(label="Estimated Price (USD)")

    btn.click(price_ui, inputs=[name_in, cat_in, brand_in, cost_in], outputs=[out_price])

# Uncomment to launch UI when running this notebook directly
# demo.launch()